In [2]:
import sqlite3
import pandas as pd

---

**Total Revenue by customer**

In [ ]:
query = """ SELECT
s.CustomerKey,
sum(OrderQuantity * ProductCost) AS Revenue
FROM Sales_Data AS s
INNER JOIN Customer_Lookup AS c
ON s.CustomerKey = c.CustomerKey
INNER JOIN Product_Lookup AS p
ON s.ProductKey = p.ProductKey
GROUP BY s.CustomerKey
ORDER BY Revenue DESC
"""

try:
    conn = sqlite3.connect('../data/AdventureWorks.db')
    rev_customer = pd.read_sql(query, conn)
except sqlite3.Error as err:
    print('Error Ocurred - ',err)
finally:
    conn.close()
rev_customer.head()

---

## SQL Magic

In [3]:
%load_ext sql
%sql sqlite:///../Data/AdventureWorks.db
# %config SqlMagic.style = '_DEPRECATED_DEFAULT'

---

**Total Return Quantity by Category**

In [23]:
%%sql
select r.ProductKey, CategoryName, SubcategoryName, sum(ReturnQuantity) as TotalReturn
from Returns_Data as r
inner join Product_Lookup as p
on r.ProductKey = p.ProductKey
inner join Product_Subcategories_Lookup as s
on p.ProductSubcategoryKey = s.ProductSubcategoryKey
inner join Product_Categories_Lookup as c
on s.ProductCategoryKey = c.ProductCategoryKey
group by r.ProductKey
order by TotalReturn desc
limit 5
;

 * sqlite:///../Data/AdventureWorks.db
Done.


ProductKey,CategoryName,SubcategoryName,TotalReturn
477,Accessories,Bottles and Cages,155
480,Accessories,Tires and Tubes,95
528,Accessories,Tires and Tubes,93
478,Accessories,Bottles and Cages,77
214,Accessories,Helmets,70


**Total Order By Territory**

In [13]:
%%sql
select Continent, Country, Region, count(distinct(OrderNumber)) as TotalOrder
from Territory_Lookup
inner join Sales_data
on TerritoryKey = SalesTerritoryKey
group by TerritoryKey
order by TotalOrder desc;

 * sqlite:///../Data/AdventureWorks.db
Done.


Continent,Country,Region,TotalOrder
Pacific,Australia,Australia,6060
North America,United States,Southwest,4992
North America,United States,Northwest,3675
North America,Canada,Canada,3024
Europe,United Kingdom,United Kingdom,2771
Europe,France,France,2315
Europe,Germany,Germany,2294
North America,United States,Southeast,14
North America,United States,Northeast,10
North America,United States,Central,9


**Months with Less than Average Order**

In [19]:
%%sql
select strftime('%m', OrderDate) as Month,
count(distinct(OrderNumber)) as TotalOrder
from Sales_Data
group by Month
having TotalOrder <
(select avg(TotalOrder) from (select strftime('%m', OrderDate) as Month,
count(distinct(OrderNumber)) as TotalOrder
from Sales_Data
group by Month));

 * sqlite:///../Data/AdventureWorks.db
Done.


Month,TotalOrder
07,753
08,1821
09,1764
10,1862
11,1868


**Return Quantity of products more than $1000**

In [5]:
%%sql
select ProductKey, sum(ReturnQuantity) as total_return
from Returns_Data as r
where exists (select ProductPrice from Product_Lookup as p
where r.ProductKey = p.ProductKey and
ProductPrice > 1000)
group by ProductKey
order by total_return desc
limit 5;

 * sqlite:///../Data/AdventureWorks.db
Done.


ProductKey,total_return
360,21
362,18
352,17
358,15
354,15


**Customers with more than Average Annual Income but Order Less Than Average Payments**

In [8]:
%%sql
select s.CustomerKey, count() as OrderTimes, AnnualIncome, OrderQuantity * ProductPrice as TotalCost
from Product_Lookup as p
inner join Sales_Data as s
on p.ProductKey = s.ProductKey
inner join Customer_Lookup as c
on s.CustomerKey = c.CustomerKey
where exists (
    select CustomerKey from Customer_Lookup as c
where c.CustomerKey = s.CustomerKey
and AnnualIncome > (select avg(AnnualIncome) from Customer_Lookup)
)
group by s.CustomerKey
having TotalCost < (select avg(OrderQuantity * ProductPrice) as avg
from Product_Lookup as p
inner join Sales_Data as s
on p.ProductKey = s.ProductKey)
order by TotalCost desc, AnnualIncome desc
limit 3;

 * sqlite:///../Data/AdventureWorks.db
Done.


CustomerKey,OrderTimes,AnnualIncome,TotalCost
15957,2,130000,159.0
21847,3,130000,159.0
24424,4,130000,159.0
